### [Random Forest for Stock Market Prediction](https://wire.insiderfinance.io/using-random-forest-for-stock-market-prediction-79cf3803c840)

In [1]:
!pip install -qq --no-deps pandas numpy yfinance scikit-learn

In [2]:
import sys

IN_COLAB = "google.colab" in sys.modules

import warnings
warnings.filterwarnings("ignore")

from IPython.display import display, HTML

import numpy as np
import pandas as pd

import yfinance as yf

np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)
pd.set_option("display.float_format", "{:.4f}".format)

%autosave 10

Autosaving every 10 seconds


#### **Collect Market Data**

In [3]:
data = yf.download("SPY", period="5y", multi_level_index=False, progress=False)
data['Returns'] = data['Close'].pct_change()

display(data.sample(20))

,Close,High,Low,Open,Volume,Returns
Date,,,,,,
2022-12-22,364.9762,370.2392,359.2722,367.2098,100120900,-0.0143
2021-08-26,418.1628,420.5991,418.0691,420.3648,57829600,-0.0059
2021-06-16,394.2435,396.8213,392.1981,396.5972,80386100,-0.0056
2021-07-02,406.4124,406.7685,403.4138,404.4915,57697700,0.0076
2025-10-31,678.1983,681.2012,675.3943,681.1614,87164100,0.0033
2022-12-06,375.8182,381.6965,373.7284,381.1526,77972200,-0.0144
2025-06-27,609.7381,611.2057,605.6925,607.7252,86258400,0.0050
2023-07-31,442.1705,442.5278,440.4898,441.8034,62040400,0.0019
2026-02-12,679.4146,693.4562,678.5170,692.3492,118829000,-0.0154


#### **Feature Engineering**

In [4]:
data['SMA20'] = data['Close'].rolling(20).mean()
data['SMA50'] = data['Close'].rolling(50).mean()
data['Momentum'] = data['Close'] - data['Close'].shift(10)
data['Volatility'] = data['Returns'].rolling(20).std()

display(data.sample(20))

,Close,High,Low,Open,Volume,Returns,SMA20,SMA50,Momentum,Volatility
Date,,,,,,,,,,
2023-07-27,437.0513,443.7642,436.1433,443.3585,92194400,-0.0066,432.5083,419.7181,2.8300,0.0052
2023-12-13,456.0498,456.3018,449.8657,450.2244,93278000,0.0138,442.5350,426.6426,15.4020,0.0044
2021-04-12,384.4648,384.7356,383.1199,383.7270,56704900,0.0004,NaN,NaN,NaN,NaN
2025-04-04,499.5533,519.9100,499.3358,517.7349,217965100,-0.0585,552.0195,574.8050,-58.0347,0.0202
2021-11-01,432.4563,433.0767,430.7266,432.7007,48433600,0.0017,420.0284,417.9894,12.0796,0.0056
2024-02-23,494.2484,496.4674,493.5185,495.6304,61321800,0.0007,482.9407,470.3178,9.2747,0.0088
2025-06-11,594.5444,598.2025,592.4781,597.3423,73658200,-0.0029,585.5065,557.4131,13.4755,0.0076
2024-09-06,529.2234,540.2318,528.3224,538.6060,68493800,-0.0168,541.2304,537.7272,-15.5333,0.0095
2025-08-25,637.0663,639.8626,636.9473,638.6231,51274300,-0.0044,631.7072,620.4752,6.4949,0.0076


#### **Define Target Variable**

In [5]:
data['Target'] = (data['Close'].shift(-1) > data['Close']).astype(int)

display(data.sample(20))

,Close,High,Low,Open,Volume,Returns,SMA20,SMA50,Momentum,Volatility,Target
Date,,,,,,,,,,,
2021-08-17,416.0826,416.9446,413.0934,416.2700,92673900,-0.0066,413.2607,405.5623,2.7080,0.0046,0
2022-11-15,380.2651,383.9104,376.4480,382.8035,93194500,0.0085,364.4749,360.9119,13.3311,0.0182,0
2021-05-27,391.6097,392.9453,391.3295,392.4317,56707700,0.0005,388.3870,NaN,8.4152,0.0090,1
2024-09-09,535.1489,536.4222,531.4958,533.4252,40445800,0.0112,541.8876,537.7280,-15.3959,0.0097,1
2025-05-02,560.3365,561.9381,556.0061,558.3295,60717300,0.0148,528.9810,549.7121,39.8926,0.0321,0
2021-07-14,408.7737,410.3480,407.5275,409.8607,64130400,0.0015,401.2703,394.8721,8.0023,0.0063,0
2025-10-07,665.3316,669.1797,663.8898,668.7322,72020100,-0.0037,658.7016,644.3818,5.8765,0.0039,1
2025-07-31,626.7637,634.4683,625.4647,634.0816,103385200,-0.0038,623.1518,604.8437,4.0060,0.0043,0
2023-08-29,433.8349,434.1150,427.3635,427.5470,83081900,0.0145,428.6230,429.7790,6.0560,0.0083,1


#### **Prepare Dataset**

In [6]:
features = ['SMA20', 'SMA50', 'Momentum', 'Volatility']
X = data[features].dropna()
y = data['Target'].loc[X.index]
print(f"len(X) = {len(X)}")
print(f"len(y) = {len(y)}")

len(X) = 1207
len(y) = 1207


#### **Train Random Forest Model**

In [7]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100)
model.fit(X, y)

RandomForestClassifier()

#### ** Make Predictions**


In [8]:
predictions = model.predict(X)

#### **Evaluate Performance**

In [9]:
from sklearn.metrics import accuracy_score
print("Accuracy:", accuracy_score(y, predictions))

Accuracy: 1.0


#### **Feature Importance**

In [10]:
importance = pd.Series(model.feature_importances_, index=features)
display(importance.sort_values(ascending=False))

,0
Momentum,0.2623
Volatility,0.2569
SMA50,0.2405
SMA20,0.2403
